# Análise de uma captura de movimento

Este notebook converte um arquivo MVNX para NPZ, carrega os dados e mostra análises básicas.

## 1. Preparação

Primeiro importamos a API do projeto e localizamos a raiz do repositório.

In [6]:
from pathlib import Path
import sys

PROJETO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))

from mvnview import MocapData
from mvnview.tools import print_all_shapes,print_joint_names, print_segment_names, save_mvnx_data, print_ergonomic_joint_relations

## 2. Localização dos arquivos

O arquivo MVNX deve estar em `capture_data/raw_mvnx/files`.
O arquivo convertido será salvo em `capture_data/converted_npz/files`.
Na próxima célula, altere somente `NOME_ARQUIVO_MVNX` e `NOME_ARQUIVO_NPZ`.

In [7]:
PASTA_ARQUIVOS_MVNX = PROJETO / "capture_data" / "raw_mvnx" / "files"
PASTA_ARQUIVOS_NPZ = PROJETO / "capture_data" / "converted_npz" / "files"
PASTA_ARQUIVOS_NPZ.mkdir(parents=True, exist_ok=True)

NOME_ARQUIVO_MVNX = "rugby_example.mvnx"
NOME_ARQUIVO_NPZ = "captura_de_movimento_convertida.npz"

arquivo_mvnx_original = PASTA_ARQUIVOS_MVNX / NOME_ARQUIVO_MVNX
arquivo_npz_convertido = PASTA_ARQUIVOS_NPZ / NOME_ARQUIVO_NPZ

if not arquivo_mvnx_original.exists():
    raise FileNotFoundError(
        f"Arquivo MVNX não encontrado: {arquivo_mvnx_original}"
    )

print(f"Arquivo original: {arquivo_mvnx_original.name}")
print(f"Arquivo convertido: {arquivo_npz_convertido.name}")

Arquivo original: rugby_example.mvnx
Arquivo convertido: captura_de_movimento_convertida.npz


## 3. Conversão do arquivo

`save_mvnx_data` lê o MVNX e salva os dados em um NPZ organizado para análises futuras.

In [8]:
save_mvnx_data(arquivo_mvnx_original, arquivo_npz_convertido)

100%|██████████| 180/180 [00:00<00:00, 52560.20it/s]


100%|██████████| 180/180 [00:00<00:00, 1168691.52it/s]


Dados e metadados salvos com sucesso em:
/home/godel/Projects/github-repos/mvnview/capture_data/converted_npz/files/captura_de_movimento_convertida.npz


## 4. Carregamento dos dados

`MocapData` abre o NPZ e oferece métodos para acessar os dados por segmento.

In [9]:
dados_captura = MocapData(arquivo_npz_convertido)

## 5. Inspeção dos dados

Primeiro vemos o formato das principais matrizes e os nomes dos segmentos disponíveis.

In [10]:
print_all_shapes(dados_captura)
print_segment_names(dados_captura)

posição: (180, 69) --> (frames, (23 segmentos x 3 [x,y,z]))
orientação: (180, 92) --> (frames, (23 segmentos x 4 [q0,q1,q2, q3]))
tempo: (180,)
segmentos: (23,)
nomes das juntas: (22,)
ângulo das juntas: (180, 66)

velocidade: (180, 69) --> (frames, (23 segmentos x 3 [x,y,z]))
aceleração: (180, 69) --> (frames, (23 segmentos x 3 [x,y,z]))
velocidade angular: (180, 69) --> (frames, (23 segmentos x 3 [x,y,z]))
aceleração angular: (180, 69) --> (frames, (23 segmentos x 3 [x,y,z]))
contatos dos pés: (180, 4) --> (frames, (23 segmentos x 3 [x,y,z]))
aceleração livre do sensor: (180, 33) --> (frames, (23 segmentos x 3 [x,y,z]))
campo magnético do sensor: (180, 33) --> (frames, (23 segmentos x 3 [x,y,z]))
orientação do sensor: (180, 44) --> (frames, (23 segmentos x 4 [q0,q1,q2, q3]))
centro de massa: (180, 9) --> (frames, (23 segmentos x 3 [x,y,z]))
Segmentos:
Pelvis         RightUpperArm  RightLowerLeg
L5             RightForeArm   RightFoot
L3             RightHand      RightToe
T12        

## 6. Consulta de um segmento

A posição de um segmento tem três colunas: X, Y e Z.

In [11]:
nome_segmento = "Head"
posicao_cabeca = dados_captura.get_segment_position(nome_segmento)
print(f"Posição de {nome_segmento} no frame primeiro frame: {posicao_cabeca[0]}")

Posição de Head no frame primeiro frame: [ 0.188283 -0.390039  1.530932]


## 7. Distância entre segmentos

A distância é calculada em cada frame entre a pelve e a cabeça.

In [12]:
distancia_pelve_cabeca = dados_captura.get_distance("Pelvis", "Head")
print(f"Quantidade de frames: {distancia_pelve_cabeca.shape[0]}")
print(f"Distância no primeiro frame: {distancia_pelve_cabeca[0]:.3f} m")

Quantidade de frames: 180
Distância no primeiro frame: 0.516 m


## 8. Ângulos e avaliações biomecânicas.

Dada a compreensão dos passos anteriores e dos gráficos presentes em **plots_simples.ipynb**, é possível utilizar as informações obtidas sobre os formatos de armazenamento dos dados para direcionar a análise e computação de informações biomecânicas. Caso seja necessário obter as **matrizes de rotação pura tridimensional (DCM)** para fazer cálculos de força, torque articular (cinética), animação 3D ou matrizes de transformação customizadas, o notebook **quaternions.ipynb** reúne funções para esse propósito. 

Caso apenas as angulações diretas sejam necessárias, a classe MocapData já encapsula os dados extraídos em:

**jointAngle**: ângulos das 22 articulações extraídos pela sequência ZXY, seguindo a convenção da ISB. É o dado padrão para membros inferiores, tronco e demais articulações.

**jointAngleXZY**: os mesmos ângulos extraídos pela sequência XZY. O MVN o exporta para todas as articulações, mas ele é comumente usado apenas nos ombros, e a adequação de seu uso depende do movimento realizado.

**jointAngleErgo** e **jointAngleErgoXZY**: ângulos de um conjunto reduzido de articulações usadas em análises ergonômicas (por exemplo, T8→Head), cujos segmentos pai e filho são definidos no cabeçalho do arquivo .mvnx.

Sobre as sequências. ZXY e XZY são ambas sequências de Tait-Bryan; mudar a ordem altera em qual ângulo ocorre a singularidade (gimbal lock, quando o ângulo intermediário se aproxima de ±90°) e como o movimento se distribui entre os canais. Por isso, a escolha entre elas deve considerar a amplitude do movimento analisado.

* Nota: O MVN User Manual pag 138/154 contêm informações a respeito de que segmentos são utilizados na comparação de ângulos entre segmentos.



Dentro do arquivo .mvnx, existem headers:
		<ergonomicJointAngles>
			<ergonomicJointAngle label="T8_Head" index="0" parentSegment="T8" childSegment="Head"/>
			<ergonomicJointAngle label="T8_LeftUpperArm" index="1" parentSegment="T8" childSegment="LeftUpperArm"/>
			<ergonomicJointAngle label="T8_RightUpperArm" index="2" parentSegment="T8" childSegment="RightUpperArm"/>
			<ergonomicJointAngle label="Pelvis_T8" index="3" parentSegment="Pelvis" childSegment="T8"/>
			<ergonomicJointAngle label="Vertical_Pelvis" index="4" parentSegment="Vertical" childSegment="Pelvis"/>
			<ergonomicJointAngle label="Vertical_T8" index="5" parentSegment="Vertical" childSegment="T8"/>
		</ergonomicJointAngles>
Eles definem para as matrizes ergonômicas (**jointAngleErgo** e **jointAngleErgoXZY**) quais são as relações

In [13]:
print_ergonomic_joint_relations(
    PASTA_ARQUIVOS_MVNX / NOME_ARQUIVO_MVNX
)

Index  Label                    Relação                          Índices das colunas na matriz
    0  T8_Head                  T8 -> Head                       0, 1, 2
    1  T8_LeftUpperArm          T8 -> LeftUpperArm               3, 4, 5
    2  T8_RightUpperArm         T8 -> RightUpperArm              6, 7, 8
    3  Pelvis_T8                Pelvis -> T8                     9, 10, 11
    4  Vertical_Pelvis          Vertical -> Pelvis               12, 13, 14
    5  Vertical_T8              Vertical -> T8                   15, 16, 17


In [14]:
print_joint_names(dados_captura)

Juntas:
jL5S1
jL4L3
jL1T12
jT9T8
jT1C7
jC1Head
jRightT4Shoulder
jRightShoulder
jRightElbow
jRightWrist
jLeftT4Shoulder
jLeftShoulder
jLeftElbow
jLeftWrist
jRightHip
jRightKnee
jRightAnkle
jRightBallFoot
jLeftHip
jLeftKnee
jLeftAnkle
jLeftBallFoot


Eixo	Movimento
Z	Flexão / Extensão
X	Abdução / Adução
Y	Rotação interna / externa

De acordo com o manual de uso "*Almost all angles follow the ISB Euler angle extractions of Z (flexion/extension), X (abduction/adduction), Y (internal/external rotation)*". Foi realizado um teste de correlação entre **jointAngle** e **jointAngleErgo**, confirmando a afirmação.

| Eixo | Movimento |
| :---: | :--- |
| **Z** | Flexão / Extensão |
| **X** | Abdução / Adução |
| **Y** | Rotação interna / externa |
